In [ ]:
# =========================================================
# CELL 1: INSTALASI DEPENDENSI (SUPER RINGAN)
# =========================================================
!pip install -q PyMuPDF tqdm beautifulsoup4 requests selenium webdriver-manager internetarchive

# Install Google Chrome (Colab base image no longer bundles it)
!apt-get update -qq
!wget -q -O /tmp/chrome.deb https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
!apt-get install -y -qq /tmp/chrome.deb
print("✅ Instalasi selesai! Silakan lanjut jalankan Cell 2.")

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
✅ Instalasi selesai! Silakan lanjut jalankan Cell 2.


In [ ]:
# =========================================================
# CELL 2: MAIN SCRIPT DENGAN DASHBOARD SWISS B&W (ENHANCED)
# =========================================================

# =========================================================
# GOOGLE DRIVE
# =========================================================
from google.colab import drive
drive.mount('/content/drive')

# =========================================================
# IMPORT
# =========================================================
import os
import re
import json
import time
import requests
import shutil
import fitz  # PyMuPDF
import internetarchive as ia
from datetime import datetime, timezone, timedelta

from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager

# Import UI Components
import ipywidgets as widgets
from IPython.display import display, HTML
from tqdm.notebook import tqdm


# =========================================================
# CONFIG
# =========================================================
QUERIES = []  # akan diisi dari kolom input keyword saat tombol MULAI ditekan
WIB = timezone(timedelta(hours=7))  # Waktu Indonesia Barat (UTC+7), server Colab biasanya UTC

ROOT_FOLDER = "/content/drive/MyDrive/COK DELPHER 3"
os.makedirs(ROOT_FOLDER, exist_ok=True)

# =========================================================
# LOGIN INTERNET ARCHIVE (upload otomatis tiap 1 buku selesai)
# =========================================================
# PENTING: JANGAN hardcode kredensial di sini. Simpan di Colab Secrets
# (ikon kunci di sidebar kiri Colab) dengan nama IA_ACCESS_KEY dan
# IA_SECRET_KEY, lalu ambil dengan userdata.get() seperti di bawah.
from google.colab import userdata
IA_ACCESS_KEY = userdata.get("IA_ACCESS_KEY")
IA_SECRET_KEY = userdata.get("IA_SECRET_KEY")

os.environ["IAS3_ACCESS_KEY"] = IA_ACCESS_KEY
os.environ["IAS3_SECRET_KEY"] = IA_SECRET_KEY

_ia_config_path = os.path.expanduser("~/.config/internetarchive/ia.ini")
os.makedirs(os.path.dirname(_ia_config_path), exist_ok=True)
with open(_ia_config_path, "w") as _f:
    _f.write(
        "[s3]\n"
        f"access = {IA_ACCESS_KEY}\n"
        f"secret = {IA_SECRET_KEY}\n"
    )

IA_UPLOAD_LOG = os.path.join(ROOT_FOLDER, "ia_upload_progress.json")
if not os.path.exists(IA_UPLOAD_LOG):
    with open(IA_UPLOAD_LOG, "w") as f:
        json.dump({"uploaded": {}}, f)  # folder_name -> archive_url

with open(IA_UPLOAD_LOG, "r") as f:
    _ia_log_raw = json.load(f).get("uploaded", {})
    # Kompatibel dengan format lama (list nama folder tanpa URL)
    if isinstance(_ia_log_raw, list):
        ia_uploaded_map = {name: None for name in _ia_log_raw}
    else:
        ia_uploaded_map = dict(_ia_log_raw)


def upload_folder_to_ia(folder_name, folder_path, detail_data):
    """Upload 1 folder buku (PDF + TXT + jpg.zip yang ada) ke archive.org
    segera setelah folder ini selesai diproses. Skip kalau folder ini
    sudah pernah berhasil diupload sebelumnya (baca ia_uploaded_map).
    Mengembalikan URL archive.org kalau sukses/sudah pernah, None kalau gagal."""
    if folder_name in ia_uploaded_map and ia_uploaded_map[folder_name]:
        return ia_uploaded_map[folder_name]

    pdf_path_local = None
    for fname in os.listdir(folder_path):
        if fname.lower().endswith(".pdf"):
            pdf_path_local = os.path.join(folder_path, fname)
            break
    if not pdf_path_local:
        return None

    files_to_upload = [pdf_path_local]
    txt_path_local = os.path.join(folder_path, "file.txt")
    if os.path.exists(txt_path_local):
        files_to_upload.append(txt_path_local)
    jpg_zip_local = os.path.join(folder_path, "jpg.zip")
    if os.path.exists(jpg_zip_local):
        files_to_upload.append(jpg_zip_local)

    safe_folder = "".join(
        c if c.isalnum() or c in "-_" else "-" for c in folder_name
    )[:80].strip("-")
    ia_identifier = f"DLP-{safe_folder}"

    subject_parts = ["delpher", "dutch colonial", "aceh", "atjeh", "indonesia"]
    keyword_local = detail_data.get("keyword", "")
    if keyword_local:
        subject_parts.insert(0, keyword_local)

    ia_metadata = {
        "title": detail_data.get("title", folder_name),
        "mediatype": "texts",
        "collection": "opensource",
        "subject": "; ".join(subject_parts),
        "language": "Dutch",
        "licenseurl": "https://creativecommons.org/licenses/by/4.0/",
        "description": (
            f"Source: Delpher (delpher.nl)\n"
            f"Original identifier: {detail_data.get('identifier', '')}\n"
            f"Detail URL: {detail_data.get('detail_url', '')}\n"
            f"Keyword pencarian: {keyword_local}"
        ),
    }

    try:
        r = ia.upload(
            ia_identifier,
            files=files_to_upload,
            metadata=ia_metadata,
            retries=3,
            retries_sleep=15,
            checksum=True,
        )
        all_ok = all(resp.status_code in [200, 201] for resp in r)
        if all_ok:
            archive_url = f"https://archive.org/details/{ia_identifier}"
            ia_uploaded_map[folder_name] = archive_url
            with open(IA_UPLOAD_LOG, "w") as f:
                json.dump({"uploaded": ia_uploaded_map}, f, indent=2, ensure_ascii=False)
            return archive_url
        return None
    except Exception:
        return None


# =========================================================
# HELPER: FORMAT SIZE & FOLDER CALCULATION
# =========================================================
def get_size_format(b, factor=1024, suffix="B"):
    """Mengubah ukuran bytes menjadi format yang mudah dibaca (KB, MB, GB, dst)"""
    for unit in ["", "K", "M", "G", "T"]:
        if b < factor:
            return f"{b:.2f} {unit}{suffix}"
        b /= factor
    return f"{b:.2f} P{suffix}"

def get_folder_size(folder_path):
    """Menghitung total ukuran semua file di dalam folder"""
    total_size = 0
    if os.path.exists(folder_path):
        for dirpath, _, filenames in os.walk(folder_path):
            for f in filenames:
                fp = os.path.join(dirpath, f)
                if not os.path.islink(fp):
                    total_size += os.path.getsize(fp)
    return total_size

def format_duration(seconds):
    """Mengubah detik menjadi format Xj Ym Zd"""
    seconds = int(max(0, seconds))
    h, rem = divmod(seconds, 3600)
    m, s = divmod(rem, 60)
    if h:
        return f"{h}j {m}m {s}d"
    if m:
        return f"{m}m {s}d"
    return f"{s}d"


# =========================================================
# PROGRESS & STATE MANAGEMENT
# =========================================================
PROGRESS_FILE = os.path.join(ROOT_FOLDER, "progress.json")

def load_progress():
    """Baca progress.json. Kembalikan struktur default jika belum ada atau korup
    (misal file sempat terputus saat ditulis sebelumnya)."""
    default = {
        "completed_identifiers": [],
        "queries": [],     # daftar keyword sesi terakhir (dipakai utk cek bisa resume atau tidak)
        "query_index": 0,  # posisi keyword terakhir yang sedang diproses
        "page": 1,         # posisi halaman terakhir yang sedang diproses
        # sengaja TIDAK ada "last_global_index" di sini -- lihat penjelasan di bawah
    }
    data = dict(default)
    if os.path.exists(PROGRESS_FILE):
        try:
            with open(PROGRESS_FILE, "r") as f:
                loaded = json.load(f)
            data.update(loaded)
        except (json.JSONDecodeError, OSError):
            # File korup -> jangan sampai crash, mulai dari default
            data = dict(default)

    # PENTING: "last_global_index" dihitung TERPISAH setelah merge, bukan diberi nilai
    # default 0 dari awal. Kalau progress.json berasal dari versi kode SEBELUM field ini
    # ada (hanya punya "completed_identifiers"), key "last_global_index" memang tidak
    # akan ditemukan di file -> di sinilah fallback ke len(completed_identifiers) dipakai,
    # supaya penomoran folder (DLP-xxx) melanjutkan dari nomor terakhir, BUKAN reset ke 0.
    # (Bug sebelumnya: default dict sudah diisi 0 duluan, jadi .get() selalu menemukan
    # key itu dengan nilai 0 dan fallback-nya tidak pernah terpakai.)
    if "last_global_index" not in data:
        data["last_global_index"] = len(data.get("completed_identifiers", []))

    return data

def save_progress():
    """Simpan progress.json secara ATOMIC: tulis ke file sementara lalu rename.
    Ini mencegah progress.json korup jika proses terputus (disconnect/crash) persis
    saat sedang menulis file."""
    data = {
        "completed_identifiers": list(completed_identifiers),
        "queries": QUERIES,
        "query_index": query_index,
        "page": page,
        "last_global_index": global_index,
    }
    tmp_path = PROGRESS_FILE + ".tmp"
    with open(tmp_path, "w") as f:
        json.dump(data, f, indent=2)
    os.replace(tmp_path, PROGRESS_FILE)  # rename adalah operasi atomic di level OS

def get_max_existing_folder_index():
    """Scan folder ROOT_FOLDER secara langsung dan cari nomor DLP-N tertinggi yang
    BENAR-BENAR ADA di disk. Ini dipakai sebagai 'ground truth' karena progress.json
    (last_global_index) bisa saja korup/salah -- misalnya kalau ada sesi sebelumnya
    yang sempat menyimpan angka kecil karena bug atau proses terputus tak terduga.
    Dengan begini penomoran folder tidak akan pernah mundur/tabrakan, apapun isi
    progress.json."""
    if not os.path.exists(ROOT_FOLDER):
        return 0
    max_index = 0
    for name in os.listdir(ROOT_FOLDER):
        m = re.match(r"^DLP-(\d+)-", name)
        if m:
            max_index = max(max_index, int(m.group(1)))
    return max_index

progress_data = load_progress()
completed_identifiers = set(progress_data.get("completed_identifiers", []))

# State resume (nilai awal, akan disesuaikan lagi saat tombol MULAI ditekan)
query_index = progress_data.get("query_index", 0)
page = progress_data.get("page", 1)

# PENTING: global_index diambil dari NILAI TERBESAR di antara 3 sumber:
#   1. last_global_index yang tersimpan di progress.json (bisa saja salah/korup)
#   2. jumlah completed_identifiers (jumlah buku yang benar-benar sukses)
#   3. nomor folder DLP-N tertinggi yang benar-benar ada di disk (ground truth paling akurat)
# Ini mencegah penomoran folder mundur ke angka kecil lagi walau progress.json
# sempat tertulis dengan nilai yang salah oleh sesi/bug sebelumnya.
global_index = max(
    progress_data.get("last_global_index", 0),
    len(completed_identifiers),
    get_max_existing_folder_index(),
)

# =========================================================
# UI DASHBOARD SETUP (SWISS DESIGN B&W)
# =========================================================
STATUS_COLORS = {
    "SUKSES": "#1a7f37",
    "SELESAI 1 BUKU": "#1a7f37",
    "TXT SUKSES": "#1a7f37",
    "PDF SUKSES": "#1a7f37",
    "ERROR": "#d1242f",
    "PDF GAGAL": "#d1242f",
    "TXT GAGAL": "#d1242f",
    "TXT ERROR": "#d1242f",
    "JPG ERROR": "#d1242f",
    "SKIP": "#6e7781",
    "DIHENTIKAN": "#b8860b",
    "MENGHENTIKAN": "#b8860b",
}

def status_color(status):
    return STATUS_COLORS.get(status, "#000000")

ui_state = {
    "keyword": "-",
    "page": 0,
    "total_downloaded": len(completed_identifiers),
    "current_title": "-",
    "current_status": "MENUNGGU KEYWORD...",
    "folder_size_bytes": get_folder_size(ROOT_FOLDER),
    "logs": [],
    "history": [],
    "success_count": 0,
    "error_count": 0,
    "session_start": time.time(),
    "book_times": [],
    "keyword_status": {},
    "keyword_counts": {},
    "stop_requested": False,
    "running": False,
    "log_filter": "",
}

out_widget = widgets.HTML(value="")

# --- Kolom input keyword (diisi sebelum run dimulai) ---
keyword_input = widgets.Text(
    value="atjeh, pidie, gajo",
    placeholder="Masukkan keyword, pisahkan dengan koma",
    description="Keyword:",
    style={"description_width": "70px"},
    layout=widgets.Layout(width="480px", height="36px")
)

start_button = widgets.Button(
    description="▶ MULAI SCRAPING",
    button_style="success",
    layout=widgets.Layout(width="180px", height="36px")
)
stop_button = widgets.Button(
    description="⏸ HENTIKAN PROSES",
    button_style="danger",
    disabled=True,
    layout=widgets.Layout(width="200px", height="36px")
)
log_filter_input = widgets.Text(
    placeholder="Filter log (mis. ERROR)",
    layout=widgets.Layout(width="240px", height="36px")
)

keyword_row = widgets.HBox([keyword_input, start_button], layout=widgets.Layout(margin="0 0 8px 0"))
action_row = widgets.HBox([stop_button, log_filter_input], layout=widgets.Layout(margin="0 0 10px 0"))
control_box = widgets.VBox([keyword_row, action_row])

def on_stop_clicked(b):
    if not ui_state["running"] or ui_state["stop_requested"]:
        return
    ui_state["stop_requested"] = True
    stop_button.description = "MENGHENTIKAN..."
    stop_button.disabled = True
    log_msg("Permintaan berhenti diterima. Menyelesaikan buku saat ini lalu berhenti...", "MENGHENTIKAN")

def on_filter_change(change):
    ui_state["log_filter"] = change["new"]
    update_ui()

stop_button.on_click(on_stop_clicked)
log_filter_input.observe(on_filter_change, names="value")

display(control_box)
display(out_widget)

def get_overall_progress():
    """Estimasi progres keseluruhan berdasarkan jumlah keyword yang selesai"""
    if not QUERIES:
        return 0
    done = sum(1 for q in QUERIES if ui_state["keyword_status"][q] == "SELESAI")
    partial = 0.5 if ui_state["keyword_status"].get(ui_state["keyword"]) == "PROSES" else 0
    return min(100, ((done + partial) / len(QUERIES)) * 100)

def get_speed_stats():
    """Kecepatan rata-rata (buku/menit) dari 20 buku terakhir"""
    times = ui_state["book_times"][-20:]
    if not times:
        return 0.0, 0.0
    avg = sum(times) / len(times)
    per_min = 60 / avg if avg > 0 else 0.0
    return per_min, avg

def update_ui():
    """Fungsi untuk merender ulang HTML bergaya Swiss B&W"""
    # 1. Cek Kapasitas Drive
    total_drive, used_drive, free_drive = shutil.disk_usage("/content/drive")
    drive_total_str = get_size_format(total_drive)
    drive_free_str = get_size_format(free_drive)

    # 2. Ukuran Folder Output
    folder_size_str = get_size_format(ui_state["folder_size_bytes"])

    # 3. Progres Keseluruhan
    progress_pct = get_overall_progress()
    progress_html = f"""
    <div style="margin-bottom: 20px;">
        <div style="display:flex; justify-content:space-between; font-size:11px; font-weight:800; text-transform:uppercase; letter-spacing:1px; margin-bottom:6px;">
            <span>Progres Keseluruhan</span><span>{progress_pct:.0f}%</span>
        </div>
        <div style="width:100%; height:14px; background:#eee; border:2px solid #000;">
            <div style="width:{progress_pct:.0f}%; height:100%; background:#000; transition: width 0.3s ease;"></div>
        </div>
    </div>
    """

    # 4. Chip Status per Keyword
    kw_chips = ""
    for q in QUERIES:
        st = ui_state["keyword_status"][q]
        cnt = ui_state["keyword_counts"][q]
        if st == "SELESAI":
            bg, fg, border, icon = "#000", "#fff", "2px solid #000", "✓ "
        elif st == "PROSES":
            bg, fg, border, icon = "#fff", "#000", "2px solid #000", "▶ "
        else:
            bg, fg, border, icon = "#fff", "#999", "2px dashed #ccc", "○ "
        kw_chips += (
            f'<span style="display:inline-block; margin-right:8px; margin-bottom:8px; '
            f'padding:6px 12px; background:{bg}; color:{fg}; border:{border}; '
            f'font-weight:800; font-size:12px; text-transform:uppercase; letter-spacing:1px;">'
            f'{icon}{q} ({cnt})</span>'
        )

    # 5. Statistik (sukses, error, kecepatan, waktu berjalan)
    per_min, avg_sec = get_speed_stats()
    elapsed = time.time() - ui_state["session_start"]
    stats_html = f"""
    <div style="display: grid; grid-template-columns: repeat(4, 1fr); gap: 12px; margin-bottom: 25px;">
        <div style="border: 2px solid #000; padding: 12px;">
            <div style="font-size: 9px; text-transform: uppercase; font-weight: bold; letter-spacing: 1px; margin-bottom: 4px;">Sukses</div>
            <div style="font-size: 20px; font-weight: 900; color: #1a7f37;">{ui_state['success_count']}</div>
        </div>
        <div style="border: 2px solid #000; padding: 12px;">
            <div style="font-size: 9px; text-transform: uppercase; font-weight: bold; letter-spacing: 1px; margin-bottom: 4px;">Error</div>
            <div style="font-size: 20px; font-weight: 900; color: #d1242f;">{ui_state['error_count']}</div>
        </div>
        <div style="border: 2px solid #000; padding: 12px;">
            <div style="font-size: 9px; text-transform: uppercase; font-weight: bold; letter-spacing: 1px; margin-bottom: 4px;">Kecepatan</div>
            <div style="font-size: 20px; font-weight: 900;">{per_min:.1f}<span style="font-size:11px; font-weight:normal;">/menit</span></div>
        </div>
        <div style="border: 2px solid #000; padding: 12px;">
            <div style="font-size: 9px; text-transform: uppercase; font-weight: bold; letter-spacing: 1px; margin-bottom: 4px;">Waktu Berjalan</div>
            <div style="font-size: 20px; font-weight: 900;">{format_duration(elapsed)}</div>
        </div>
    </div>
    """

    # 6. Render Baris History (Tabel) dengan badge status berwarna
    history_html = ""
    for h in ui_state["history"]:
        c = status_color(h['status'])
        history_html += f"""
        <tr>
            <td style="border-bottom: 1px solid #000; padding: 10px; font-weight: bold; text-align: center;">{h['number']}</td>
            <td style="border-bottom: 1px solid #000; padding: 10px; font-weight: bold;">{h['keyword']}</td>
            <td style="border-bottom: 1px solid #000; padding: 10px;">{h['identifier']}</td>
            <td style="border-bottom: 1px solid #000; padding: 10px; white-space: nowrap; overflow: hidden; text-overflow: ellipsis; max-width: 220px;">{h['title']}</td>
            <td style="border-bottom: 1px solid #000; padding: 10px;">
                <span style="display:inline-block; padding: 3px 8px; border: 2px solid {c}; color: {c}; font-weight: 900; text-transform: uppercase; font-size: 11px; letter-spacing: 0.5px;">{h['status']}</span>
            </td>
            <td style="border-bottom: 1px solid #000; padding: 10px; white-space: nowrap;">{h['size']}</td>
        </tr>
        """
    if not ui_state["history"]:
        history_html = "<tr><td colspan='6' style='padding:10px; text-align:center;'>Belum ada data di sesi ini</td></tr>"

    # 7. Render Log (dengan filter)
    filtered_logs = ui_state["logs"]
    if ui_state["log_filter"]:
        needle = ui_state["log_filter"].lower()
        filtered_logs = [l for l in ui_state["logs"] if needle in l.lower()]
    if filtered_logs:
        logs_html = "<br>".join(filtered_logs[-8:])
    else:
        logs_html = "<i>Tidak ada log yang cocok dengan filter.</i>" if ui_state["log_filter"] else "-"

    # 8. Badge status current action, ikut warna
    current_status_color = status_color(ui_state["current_status"])
    current_status_bg = current_status_color if current_status_color != "#000000" else "#000"

    # 9. Build HTML (Swiss Style)
    html = f"""
    <div style="font-family: 'Helvetica Neue', Helvetica, Arial, sans-serif; background: #fff; color: #000; padding: 25px; border: 4px solid #000; max-width: 900px; box-sizing: border-box;">

        <!-- Header -->
        <div style="border-bottom: 5px solid #000; padding-bottom: 10px; margin-bottom: 20px; display: flex; justify-content: space-between; align-items: flex-end;">
            <h1 style="margin: 0; font-size: 28px; text-transform: uppercase; font-weight: 900; letter-spacing: -1px;">Delpher Scraper</h1>
            <span style="font-weight: 800; font-size: 12px; text-transform: uppercase; background: #000; color: #fff; padding: 4px 10px; letter-spacing: 1px;">Sistem Aktif</span>
        </div>

        <!-- Keyword Chips -->
        <div style="margin-bottom: 20px;">
            {kw_chips}
        </div>

        <!-- Progres Keseluruhan -->
        {progress_html}

        <!-- Storage Info Grid -->
        <div style="display: grid; grid-template-columns: 1fr 1fr; gap: 20px; margin-bottom: 20px;">
            <div style="border: 2px solid #000; padding: 15px;">
                <div style="font-size: 10px; text-transform: uppercase; font-weight: bold; letter-spacing: 2px; margin-bottom: 5px; color: #000;">Sisa Kapasitas Drive</div>
                <div style="font-size: 26px; font-weight: 900; letter-spacing: -1px;">{drive_free_str} <span style="font-size:14px; font-weight:normal; letter-spacing:0;">/ {drive_total_str}</span></div>
            </div>
            <div style="border: 2px solid #000; padding: 15px;">
                <div style="font-size: 10px; text-transform: uppercase; font-weight: bold; letter-spacing: 2px; margin-bottom: 5px; color: #000;">Ukuran Output Folder</div>
                <div style="font-size: 26px; font-weight: 900; letter-spacing: -1px;">{folder_size_str}</div>
            </div>
        </div>

        <!-- Stats Grid -->
        {stats_html}

        <!-- Current Action Panel -->
        <div style="border: 2px solid #000; padding: 15px; margin-bottom: 25px; background: #f4f4f4;">
            <div style="font-size: 10px; text-transform: uppercase; font-weight: bold; letter-spacing: 2px; margin-bottom: 10px;">Proses Berjalan Saat Ini</div>
            <div style="font-size: 16px; font-weight: bold; margin-bottom: 5px; white-space: nowrap; overflow: hidden; text-overflow: ellipsis;">
                {ui_state['current_title']}
            </div>
            <div style="font-size: 14px; margin-bottom: 12px;">
                KEYWORD: <strong>{ui_state['keyword']}</strong> &nbsp;|&nbsp; PAGE: <strong>{ui_state['page']}</strong>
            </div>
            <div style="font-size: 12px; background: {current_status_bg}; color: #fff; display: inline-block; padding: 6px 12px; font-weight: bold; text-transform: uppercase; letter-spacing: 1px;">
                {ui_state['current_status']}
            </div>
        </div>

        <!-- Tabel Riwayat -->
        <table style="width: 100%; border-collapse: collapse; margin-bottom: 25px; border-top: 3px solid #000;">
            <thead>
                <tr>
                    <th style="background: #000; color: #fff; padding: 12px 8px; text-align: center; text-transform: uppercase; font-size: 11px; letter-spacing: 1px; width: 6%;">No</th>
                    <th style="background: #000; color: #fff; padding: 12px 10px; text-align: left; text-transform: uppercase; font-size: 11px; letter-spacing: 1px; width: 12%;">Keyword</th>
                    <th style="background: #000; color: #fff; padding: 12px 10px; text-align: left; text-transform: uppercase; font-size: 11px; letter-spacing: 1px; width: 20%;">Identifier</th>
                    <th style="background: #000; color: #fff; padding: 12px 10px; text-align: left; text-transform: uppercase; font-size: 11px; letter-spacing: 1px; width: 32%;">Judul Buku</th>
                    <th style="background: #000; color: #fff; padding: 12px 10px; text-align: left; text-transform: uppercase; font-size: 11px; letter-spacing: 1px; width: 15%;">Status</th>
                    <th style="background: #000; color: #fff; padding: 12px 10px; text-align: left; text-transform: uppercase; font-size: 11px; letter-spacing: 1px; width: 15%;">Ukuran</th>
                </tr>
            </thead>
            <tbody style="font-size: 13px;">
                {history_html}
            </tbody>
        </table>

        <!-- Log Viewer -->
        <div style="font-size: 10px; text-transform: uppercase; font-weight: bold; letter-spacing: 2px; margin-bottom: 5px;">Log Sistem</div>
        <div style="border: 2px solid #000; padding: 12px; font-family: 'Courier New', Courier, monospace; font-size: 12px; height: 140px; overflow-y: auto; background: #fff; color: #000; line-height: 1.4;">
            {logs_html}
        </div>
    </div>
    """
    out_widget.value = html

def log_msg(msg, status=None):
    """Menambah log dan merender ulang UI"""
    if status:
        ui_state["current_status"] = status

    ts = datetime.now(WIB).strftime("%d-%m-%Y %H:%M:%S")  # tanggal + jam WIB (UTC+7), bukan waktu server
    ui_state["logs"].append(f"<b>[{ts}]</b> {msg}")

    if len(ui_state["logs"]) > 50:
        ui_state["logs"].pop(0)

    update_ui()

def add_history(identifier, title, status, number="-", size_str="-"):
    """Menambah riwayat buku ke tabel UI"""
    ui_state["history"].insert(0, {
        "number": number,
        "keyword": ui_state["keyword"],
        "identifier": identifier,
        "title": title,
        "status": status,
        "size": size_str
    })
    if len(ui_state["history"]) > 4:  # Tampilkan maksimal 4 riwayat terakhir agar UI tidak terlalu panjang
        ui_state["history"].pop()

def render_summary():
    """Kartu ringkasan akhir sesi (ditampilkan setelah proses berhenti/selesai)"""
    elapsed = time.time() - ui_state["session_start"]
    per_min, avg_sec = get_speed_stats()
    html = f"""
    <div style="font-family: 'Helvetica Neue', Helvetica, Arial, sans-serif; background: #000; color: #fff; padding: 30px; border: 4px solid #000; max-width: 900px; box-sizing: border-box;">
        <div style="font-size: 11px; text-transform: uppercase; letter-spacing: 2px; font-weight: 800; margin-bottom: 6px; opacity: 0.7;">Sesi Berakhir</div>
        <h1 style="margin: 0 0 20px 0; font-size: 26px; text-transform: uppercase; font-weight: 900;">Ringkasan Scraping</h1>
        <div style="display: grid; grid-template-columns: repeat(2, 1fr); gap: 16px;">
            <div style="border: 2px solid #fff; padding: 15px;">
                <div style="font-size: 10px; text-transform: uppercase; letter-spacing: 1px; opacity: 0.7; margin-bottom: 6px;">Total Buku Sukses</div>
                <div style="font-size: 28px; font-weight: 900;">{ui_state['success_count']}</div>
            </div>
            <div style="border: 2px solid #fff; padding: 15px;">
                <div style="font-size: 10px; text-transform: uppercase; letter-spacing: 1px; opacity: 0.7; margin-bottom: 6px;">Total Error</div>
                <div style="font-size: 28px; font-weight: 900; color: #ff6b6b;">{ui_state['error_count']}</div>
            </div>
            <div style="border: 2px solid #fff; padding: 15px;">
                <div style="font-size: 10px; text-transform: uppercase; letter-spacing: 1px; opacity: 0.7; margin-bottom: 6px;">Ukuran Folder</div>
                <div style="font-size: 28px; font-weight: 900;">{get_size_format(get_folder_size(ROOT_FOLDER))}</div>
            </div>
            <div style="border: 2px solid #fff; padding: 15px;">
                <div style="font-size: 10px; text-transform: uppercase; letter-spacing: 1px; opacity: 0.7; margin-bottom: 6px;">Total Waktu</div>
                <div style="font-size: 28px; font-weight: 900;">{format_duration(elapsed)}</div>
            </div>
        </div>
        <div style="margin-top: 20px; font-size: 12px; opacity: 0.6;">Rata-rata {avg_sec:.1f} detik/buku &nbsp;|&nbsp; {per_min:.1f} buku/menit</div>
    </div>
    """
    out_widget.value = html


# =========================================================
# FUNGSI UTAMA: DIPANGGIL SAAT TOMBOL MULAI DITEKAN
# =========================================================
def cleanup_orphan_folders():
    """Hapus folder buku yang belum selesai lengkap (sisa dari proses yang terputus
    di tengah jalan, misal disconnect saat mengunduh PDF). Folder dianggap VALID
    hanya jika identifier-nya sudah tercatat sebagai completed. Ini mencegah folder
    setengah jadi menumpuk di Drive dan mencegah tabrakan penamaan folder."""
    if not os.path.exists(ROOT_FOLDER):
        return
    removed = 0
    for name in os.listdir(ROOT_FOLDER):
        folder_path = os.path.join(ROOT_FOLDER, name)
        if not os.path.isdir(folder_path) or not name.startswith("DLP-"):
            continue
        detail_path = os.path.join(folder_path, "detail.json")
        identifier = None
        if os.path.exists(detail_path):
            try:
                with open(detail_path, "r", encoding="utf-8") as f:
                    identifier = json.load(f).get("identifier")
            except (json.JSONDecodeError, OSError):
                identifier = None
        if identifier is None or identifier not in completed_identifiers:
            shutil.rmtree(folder_path, ignore_errors=True)
            removed += 1
    if removed:
        log_msg(f"Membersihkan {removed} folder buku yang belum selesai (sisa proses yang sempat terputus).", "PEMBERSIHAN AWAL")


def run_scraper():
    global global_index, query_index, page

    log_msg("Memeriksa folder yang belum selesai dari sesi sebelumnya...", "PEMBERSIHAN AWAL")
    cleanup_orphan_folders()

    log_msg("Menyiapkan Chrome Headless...", "STARTING CHROME")
    options = Options()
    options.binary_location = "/usr/bin/google-chrome"  # point Selenium at the installed Chrome binary
    options.add_argument("--headless=new")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--disable-gpu")

    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

    session = requests.Session()
    headers = {"User-Agent": "Mozilla/5.0"}

    MASTER_JSON = os.path.join(ROOT_FOLDER, "MASTER_DATA.json")
    if not os.path.exists(MASTER_JSON):
        with open(MASTER_JSON, "w") as f:
            json.dump([], f)

    # CATATAN: query_index, page, dan global_index TIDAK di-reset paksa di sini lagi.
    # Nilainya sudah ditentukan oleh on_start_clicked() -- baik itu 0/1 (mulai baru)
    # atau nilai resume dari sesi sebelumnya yang terputus (lihat on_start_clicked).
    log_msg(f"Setup selesai. Melanjutkan dari keyword ke-{query_index + 1}, halaman {page}, nomor urut buku #{global_index}.", "MENUNGGU")

    # =========================================================
    # KEYWORD & PAGE LOOP
    # =========================================================
    while True:
        if ui_state["stop_requested"]:
            log_msg("Proses dihentikan oleh pengguna.", "DIHENTIKAN")
            break

        if query_index >= len(QUERIES):
            log_msg("SEMUA KEYWORD SELESAI DIPROSES!", "SELESAI 100%")
            ui_state["current_title"] = "-"
            ui_state["keyword"] = "SELESAI"
            update_ui()
            break

        current_query = QUERIES[query_index]
        ui_state["keyword"] = current_query
        ui_state["page"] = page
        ui_state["keyword_status"][current_query] = "PROSES"
        update_ui()

        BASE_URL = f"https://www.delpher.nl/nl/boeken/results?query={current_query}&page={{}}&coll=boeken"
        search_url = BASE_URL.format(page)

        log_msg(f"Mencari URL: {search_url}", "MENCARI HALAMAN")

        driver.get(search_url)
        time.sleep(5)

        soup = BeautifulSoup(driver.page_source, "html.parser")
        page_links = []

        for a in soup.find_all("a", href=True):
            href = a["href"]
            if "/nl/boeken/objectsearch/pagejump" in href:
                if href.startswith("/"):
                    href = "https://www.delpher.nl" + href
                page_links.append(href)

        page_links = list(dict.fromkeys(page_links))
        log_msg(f"Ditemukan {len(page_links)} buku.", "MEMERIKSA LINK")

        if len(page_links) == 0:
            log_msg(f"Tidak ada buku lagi untuk keyword '{current_query}'.", "PINDAH KEYWORD")
            ui_state["keyword_status"][current_query] = "SELESAI"
            query_index += 1
            page = 1
            save_progress()  # simpan titik resume: pindah ke keyword berikutnya
            continue

        # =====================================================
        # PROCESS BOOKS
        # =====================================================
        for url in page_links:
            if ui_state["stop_requested"]:
                log_msg("Menghentikan sebelum buku berikutnya diproses.", "DIHENTIKAN")
                break

            book_start_time = time.time()
            identifier = None
            try:
                match = re.search(r'identifier=([^&]+)', url)
                if match:
                    identifier = match.group(1)

                if not identifier:
                    continue

                if identifier in completed_identifiers:
                    log_msg(f"ID {identifier} sudah ada. Skip.", "SKIP")
                    continue

                global_index += 1
                log_msg(f"Membuka detail buku...", "AKSES DETAIL BUKU")

                driver.get(url)
                time.sleep(5)
                detail_soup = BeautifulSoup(driver.page_source, "html.parser")

                # Ekstrak Judul
                title_tag = detail_soup.find("h1")
                title = title_tag.get_text(" ", strip=True) if title_tag else f"BOOK_{global_index}"

                ui_state["current_title"] = title
                log_msg(f"Memproses: {title[:40]}...", "EKSTRAK DATA")

                safe_title = re.sub(r'[^a-zA-Z0-9]+', '_', title).strip("_")[:120]
                folder_name = f"DLP-{global_index}-{safe_title}"
                book_folder = os.path.join(ROOT_FOLDER, folder_name)
                os.makedirs(book_folder, exist_ok=True)

                # Flag status per-buku -- dipakai untuk menentukan isi process.txt
                # dan apakah buku ini boleh ditandai SELESAI atau harus dicoba lagi
                txt_ok = False
                pdf_ok = False
                jpg_ok = False

                # Simpan HTML
                with open(os.path.join(book_folder, "file.html"), "w", encoding="utf-8") as f:
                    f.write(driver.page_source)

                # URL Endpoint API
                pdf_url = f"https://www.delpher.nl/nl/api/resource?identifier={identifier}&coll=boeken&operation=download&type=pdf"
                txt_url = f"https://www.delpher.nl/nl/api/resource?identifier={identifier}&coll=boeken&operation=download&type=objectocr"

                # =============================================
                # DOWNLOAD TXT FILE (OCR)
                # =============================================
                log_msg("Mengunduh file teks (OCR)...", "UNDUH TXT")
                txt_filename = "file.txt"
                txt_path = os.path.join(book_folder, txt_filename)
                try:
                    r_txt = session.get(txt_url, headers=headers, timeout=120)
                    if r_txt.status_code == 200:
                        with open(txt_path, "w", encoding="utf-8") as f:
                            f.write(r_txt.text)
                        log_msg("File TXT berhasil diunduh.", "TXT SUKSES")
                        txt_ok = True
                    else:
                        log_msg(f"TXT Error Status: {r_txt.status_code}", "TXT GAGAL")
                except Exception as e:
                    log_msg(f"Gagal unduh TXT: {e}", "TXT ERROR")


                # =============================================
                # DOWNLOAD PDF & EKSTRAKSI JPG
                # =============================================
                log_msg("Mengunduh PDF...", "UNDUH PDF")
                pdf_filename = f"DLP-{global_index}-{safe_title}.pdf"
                pdf_path = os.path.join(book_folder, pdf_filename)

                r = session.get(pdf_url, headers=headers, stream=True, timeout=120)
                if r.status_code == 200 and "pdf" in r.headers.get("Content-Type", "").lower():
                    total_size = int(r.headers.get("content-length", 0))

                    with open(pdf_path, "wb") as f:
                        with tqdm(total=total_size, unit='B', unit_scale=True, desc=f"UNDUH PDF", leave=False) as pbar:
                            for chunk in r.iter_content(8192):
                                if chunk:
                                    f.write(chunk)
                                    pbar.update(len(chunk))
                    log_msg("PDF berhasil disimpan.", "PDF SUKSES")
                    pdf_ok = True

                    # =============================================
                    # EXTRACT PDF TO JPG -> ZIP
                    # =============================================
                    log_msg("Mengekstrak PDF menjadi JPG...", "EKSTRAK JPG")
                    jpg_folder = os.path.join(book_folder, "jpg")
                    os.makedirs(jpg_folder, exist_ok=True)

                    try:
                        pdf_doc = fitz.open(pdf_path)
                        for page_num in tqdm(range(len(pdf_doc)), desc="EKSTRAK JPG", leave=False):
                            pdf_page = pdf_doc.load_page(page_num)
                            pix = pdf_page.get_pixmap(dpi=150)
                            jpg_filepath = os.path.join(jpg_folder, f"page_{page_num + 1:04d}.jpg")
                            pix.save(jpg_filepath)
                        pdf_doc.close()
                        log_msg("Gambar sukses diekstrak.", "MEMBUAT ZIP")

                        zip_base_name = os.path.join(book_folder, "jpg")
                        shutil.make_archive(zip_base_name, 'zip', jpg_folder)
                        log_msg("Folder JPG dibungkus ke ZIP.", "PEMBERSIHAN")

                        shutil.rmtree(jpg_folder)
                        jpg_ok = True
                    except Exception as e:
                        log_msg(f"Gagal Ekstrak/Zip: {e}", "JPG ERROR")
                else:
                    log_msg("Gagal mengunduh PDF.", "PDF GAGAL")


                # =============================================
                # PENYIMPANAN DATA MASTER & UPDATE STATUS
                # =============================================
                detail_data = {
                    "title": title,
                    "keyword": ui_state["keyword"],
                    "identifier": identifier,
                    "detail_url": url,
                    "txt_url": txt_url,
                    "pdf_url": pdf_url,
                    "folder": folder_name
                }

                with open(os.path.join(book_folder, "detail.json"), "w", encoding="utf-8") as f:
                    json.dump(detail_data, f, ensure_ascii=False, indent=2)

                if pdf_ok:
                    # ---------------------------------------------------
                    # BUKU DIANGGAP SELESAI hanya kalau PDF berhasil diunduh.
                    # TXT/JPG opsional -- statusnya tetap dicatat di process.txt,
                    # tapi tidak menggagalkan keseluruhan buku.
                    # ---------------------------------------------------
                    log_msg("Menyimpan log & metadata...", "MENYIMPAN DATA")

                    process_text = (
                        f"STATUS: SUCCESS\n"
                        f"TITLE: {title}\n"
                        f"KEYWORD: {ui_state['keyword']}\n"
                        f"IDENTIFIER: {identifier}\n"
                        f"TXT FILE: {txt_filename} ({'OK' if txt_ok else 'GAGAL - tidak tersedia'})\n"
                        f"PDF FILE: {pdf_filename} (OK)\n"
                        f"ZIP: jpg.zip ({'OK' if jpg_ok else 'GAGAL - tidak tersedia'})\n"
                        f"WAKTU: {datetime.now(WIB).strftime('%d-%m-%Y %H:%M:%S')} WIB"
                    )
                    with open(os.path.join(book_folder, "process.txt"), "w", encoding="utf-8") as f:
                        f.write(process_text)

                    # Update Tracker ID DULU (sebelum tulis Master JSON). Kalau proses terputus
                    # tepat di titik ini, buku ini otomatis dianggap SUDAH SELESAI saat run
                    # berikutnya (tidak diproses ulang) -- lebih aman daripada berisiko menulis
                    # entri dobel ke MASTER_DATA.json.
                    completed_identifiers.add(identifier)
                    save_progress()  # simpan atomic (completed_identifiers + posisi query_index/page/global_index)

                    # Master JSON -- cek duplikat by identifier dulu, jaga-jaga kalau proses
                    # sebelumnya sempat menulis ke MASTER_DATA.json tapi terputus sebelum
                    # progress.json sempat tersimpan (skenario langka, tapi dicegah di sini)
                    with open(MASTER_JSON, "r", encoding="utf-8") as f:
                        master_data = json.load(f)
                    if not any(entry.get("identifier") == identifier for entry in master_data):
                        master_data.append(detail_data)
                        with open(MASTER_JSON, "w", encoding="utf-8") as f:
                            json.dump(master_data, f, ensure_ascii=False, indent=2)

                    # Hitung ukuran folder buku ini SEKALI saja (PDF + TXT + ZIP JPG + HTML)
                    book_size_bytes = get_folder_size(book_folder)
                    book_size_str = get_size_format(book_size_bytes)

                    # Update ukuran folder ROOT secara INKREMENTAL (tambahkan ukuran buku baru)
                    # Menghindari os.walk() ulang ke seluruh ROOT_FOLDER setiap 1 buku selesai,
                    # yang tadinya membuat proses makin lambat seiring makin banyak buku terkumpul.
                    ui_state["folder_size_bytes"] += book_size_bytes

                    # Statistik: sukses, kecepatan, per-keyword
                    ui_state["success_count"] += 1
                    ui_state["keyword_counts"][current_query] += 1
                    ui_state["book_times"].append(time.time() - book_start_time)

                    # Memasukkan ke tabel history
                    add_history(identifier, title, "SUKSES", number=global_index, size_str=book_size_str)

                    ui_state["total_downloaded"] = len(completed_identifiers)
                    log_msg(f"Buku berhasil diproses & disimpan.", "SELESAI 1 BUKU")

                    # =============================================
                    # UPLOAD LANGSUNG KE INTERNET ARCHIVE
                    # (dilakukan sekarang, tiap 1 folder selesai --
                    # tidak menunggu semua buku selesai discrape)
                    # =============================================
                    log_msg("Mengupload ke Internet Archive...", "UPLOAD IA")
                    ia_url = upload_folder_to_ia(folder_name, book_folder, detail_data)

                    if ia_url:
                        log_msg(f"Upload IA sukses: {ia_url}", "UPLOAD IA SUKSES")

                        # Catat URL archive.org ke detail.json folder ini
                        detail_data["archive_url"] = ia_url
                        with open(os.path.join(book_folder, "detail.json"), "w", encoding="utf-8") as f:
                            json.dump(detail_data, f, ensure_ascii=False, indent=2)

                        # Catat juga ke process.txt biar kelihatan langsung tanpa buka detail.json
                        with open(os.path.join(book_folder, "process.txt"), "a", encoding="utf-8") as f:
                            f.write(f"\nARCHIVE.ORG: {ia_url}")

                        # Update entri buku ini di MASTER_DATA.json dengan archive_url
                        with open(MASTER_JSON, "r", encoding="utf-8") as f:
                            master_data = json.load(f)
                        for entry in master_data:
                            if entry.get("identifier") == identifier:
                                entry["archive_url"] = ia_url
                        with open(MASTER_JSON, "w", encoding="utf-8") as f:
                            json.dump(master_data, f, ensure_ascii=False, indent=2)
                    else:
                        log_msg("Upload IA gagal, akan dicoba lagi run berikutnya.", "UPLOAD IA GAGAL")

                else:
                    # ---------------------------------------------------
                    # PDF GAGAL -> buku TIDAK ditandai selesai, supaya otomatis
                    # dicoba lagi di run berikutnya. Folder ini akan dibersihkan
                    # oleh cleanup_orphan_folders() di awal run selanjutnya.
                    # ---------------------------------------------------
                    process_text = (
                        f"STATUS: FAILED\n"
                        f"TITLE: {title}\n"
                        f"KEYWORD: {ui_state['keyword']}\n"
                        f"IDENTIFIER: {identifier}\n"
                        f"TXT FILE: {txt_filename} ({'OK' if txt_ok else 'GAGAL - tidak tersedia'})\n"
                        f"PDF FILE: GAGAL DIUNDUH\n"
                        f"ZIP: tidak dibuat (PDF gagal)\n"
                        f"WAKTU: {datetime.now(WIB).strftime('%d-%m-%Y %H:%M:%S')} WIB\n"
                        f"CATATAN: Buku ini akan otomatis dicoba lagi di run berikutnya."
                    )
                    with open(os.path.join(book_folder, "process.txt"), "w", encoding="utf-8") as f:
                        f.write(process_text)

                    save_progress()  # tetap simpan posisi global_index terkini walau buku gagal

                    ui_state["error_count"] += 1
                    ui_state["book_times"].append(time.time() - book_start_time)
                    add_history(identifier, title, "PDF GAGAL", number=global_index)
                    log_msg("Buku gagal diproses (PDF tidak berhasil diunduh). Akan dicoba lagi run berikutnya.", "PDF GAGAL")

            except Exception as e:
                ui_state["error_count"] += 1
                ui_state["book_times"].append(time.time() - book_start_time)
                add_history(identifier if identifier else "UNKNOWN", str(e)[:30], "ERROR", number=global_index)
                log_msg(f"ERROR TERJADI: {e}", "ERROR")

                # Kalau folder buku ini sempat terbentuk sebelum exception terjadi,
                # tulis process.txt supaya ada jejak penyebab kegagalan (berguna untuk
                # debugging manual walau folder ini nanti akan dibersihkan otomatis
                # oleh cleanup_orphan_folders() di run berikutnya).
                if 'book_folder' in locals() and os.path.exists(book_folder):
                    try:
                        process_text = (
                            f"STATUS: FAILED\n"
                            f"TITLE: {locals().get('title', '-')}\n"
                            f"KEYWORD: {ui_state['keyword']}\n"
                            f"IDENTIFIER: {identifier if identifier else '-'}\n"
                            f"ERROR: {e}\n"
                            f"WAKTU: {datetime.now(WIB).strftime('%d-%m-%Y %H:%M:%S')} WIB\n"
                            f"CATATAN: Buku ini akan otomatis dicoba lagi di run berikutnya."
                        )
                        with open(os.path.join(book_folder, "process.txt"), "w", encoding="utf-8") as f:
                            f.write(process_text)
                    except Exception:
                        pass  # jangan sampai gagal tulis process.txt bikin scraper ikut crash

                save_progress()  # simpan posisi global_index terkini walau buku ini gagal,
                                 # supaya nomor urut tidak tabrakan kalau proses terputus setelah ini
                continue

        page += 1
        save_progress()  # simpan titik resume: pindah ke halaman berikutnya

    driver.quit()
    log_msg("Skrip ditutup dengan aman.", "SYSTEM HALTED")
    render_summary()

    # Kembalikan kontrol UI ke kondisi siap-mulai
    ui_state["running"] = False
    start_button.disabled = False
    start_button.description = "▶ MULAI SCRAPING"
    keyword_input.disabled = False
    stop_button.disabled = True
    stop_button.description = "⏸ HENTIKAN PROSES"


def on_start_clicked(b):
    global QUERIES, query_index, page

    if ui_state["running"]:
        return

    raw_keywords = [k.strip() for k in keyword_input.value.split(",") if k.strip()]
    if not raw_keywords:
        log_msg("Masukkan minimal satu keyword sebelum memulai.", "INPUT KOSONG")
        return

    QUERIES = raw_keywords

    # Inisialisasi status keyword LEBIH DULU, sebelum log resume dipanggil di bawah -
    # supaya update_ui()/get_overall_progress() tidak KeyError saat mencari status keyword.
    ui_state["keyword_status"] = {q: "MENUNGGU" for q in QUERIES}
    ui_state["keyword_counts"] = {q: 0 for q in QUERIES}

    # Resume otomatis: kalau daftar keyword PERSIS SAMA dengan sesi terakhir yang
    # tersimpan di progress.json (dan sesi itu belum selesai), lanjutkan dari
    # keyword & halaman terakhir alih-alih mengulang scan dari keyword pertama.
    # Kalau keyword berbeda (user ganti input), mulai dari awal seperti biasa.
    sesi_bisa_dilanjutkan = (
        progress_data.get("queries") == QUERIES
        and (progress_data.get("query_index", 0) > 0 or progress_data.get("page", 1) > 1)
    )
    if sesi_bisa_dilanjutkan:
        query_index = progress_data.get("query_index", 0)
        page = progress_data.get("page", 1)
        if query_index < len(QUERIES):
            log_msg(
                f"Melanjutkan sesi sebelumnya yang terputus: keyword '{QUERIES[query_index]}', halaman {page}.",
                "RESUME"
            )
    else:
        query_index = 0
        page = 1

    # Reset statistik untuk sesi run baru (jumlah buku selesai sebelumnya tetap tersimpan di progress.json)
    ui_state["success_count"] = 0
    ui_state["error_count"] = 0
    ui_state["book_times"] = []
    ui_state["history"] = []
    ui_state["logs"] = []
    ui_state["stop_requested"] = False
    ui_state["session_start"] = time.time()
    ui_state["running"] = True
    ui_state["current_title"] = "-"

    keyword_input.disabled = True
    start_button.disabled = True
    start_button.description = "SEDANG BERJALAN..."
    stop_button.disabled = False

    run_scraper()

start_button.on_click(on_start_clicked)
_resume_info = ""
if progress_data.get("queries") and (progress_data.get("query_index", 0) > 0 or progress_data.get("page", 1) > 1):
    _resume_info = (
        f" Ditemukan sesi terputus: keyword {progress_data.get('queries')}, "
        f"posisi ke-{progress_data.get('query_index', 0) + 1}, halaman {progress_data.get('page', 1)}. "
        f"Masukkan keyword yang SAMA untuk melanjutkan otomatis."
    )
log_msg(f"Siap memulai. {len(completed_identifiers)} buku sudah pernah diunduh sebelumnya.{_resume_info}", "MENUNGGU KEYWORD")


In [ ]:
# =========================================================
# CELL 3 (OPSIONAL - BACKFILL): UPLOAD ULANG FOLDER LAMA
# Cell 2 sekarang sudah otomatis upload ke Internet Archive
# TIAP 1 FOLDER SELESAI. Cell ini HANYA untuk backfill folder
# yang dibuat SEBELUM update ini, atau yang upload-nya sempat
# gagal (koneksi putus dll). Aman dijalankan berkali-kali,
# folder yang sudah ada di ia_upload_progress.json akan di-skip.
# =========================================================

# =========================================================
# INSTALL LIBRARY
# =========================================================
!pip install -q internetarchive tqdm

# =========================================================
# IMPORT
# =========================================================
import os
import json
import time
import internetarchive as ia
from tqdm.notebook import tqdm

# =========================================================
# LOGIN INTERNET ARCHIVE
# =========================================================
IA_ACCESS_KEY = "bxObWiXNLQUlcZpQ"
IA_SECRET_KEY = "cBRXZYxzlfzhXqqQ"

os.environ["IAS3_ACCESS_KEY"] = IA_ACCESS_KEY
os.environ["IAS3_SECRET_KEY"] = IA_SECRET_KEY

ia_config_path = os.path.expanduser("~/.config/internetarchive/ia.ini")
os.makedirs(os.path.dirname(ia_config_path), exist_ok=True)
with open(ia_config_path, "w") as _f:
    _f.write(
        "[s3]\n"
        f"access = {IA_ACCESS_KEY}\n"
        f"secret = {IA_SECRET_KEY}\n"
    )

print("LOGIN OK")

# =========================================================
# CONFIG
# (pakai ROOT_FOLDER yang sama dengan Cell 2, tidak perlu mount ulang Drive)
# =========================================================
print("ROOT:", ROOT_FOLDER)

# =========================================================
# PROGRESS UPLOAD (file terpisah dari progress.json scraping,
# supaya proses upload bisa dihentikan/dilanjutkan tanpa mengganggu
# progress scraping)
# =========================================================
IA_PROGRESS_FILE = os.path.join(ROOT_FOLDER, "ia_upload_progress.json")

if not os.path.exists(IA_PROGRESS_FILE):
    with open(IA_PROGRESS_FILE, "w") as f:
        json.dump({"uploaded": {}}, f)  # folder_name -> archive_url

with open(IA_PROGRESS_FILE, "r") as f:
    _raw = json.load(f).get("uploaded", {})
    # Kompatibel dengan format lama (list nama folder tanpa URL)
    ia_uploaded_map = {name: None for name in _raw} if isinstance(_raw, list) else dict(_raw)

print(f"SUDAH UPLOAD: {len(ia_uploaded_map)} item")

MASTER_JSON = os.path.join(ROOT_FOLDER, "MASTER_DATA.json")
if not os.path.exists(MASTER_JSON):
    with open(MASTER_JSON, "w") as f:
        json.dump([], f)

# =========================================================
# SCAN SEMUA SUBFOLDER DLP-*
# =========================================================
dlp_folders = []

for name in sorted(os.listdir(ROOT_FOLDER)):
    full_path = os.path.join(ROOT_FOLDER, name)
    if os.path.isdir(full_path) and name.startswith("DLP-"):
        dlp_folders.append((name, full_path))

print(f"TOTAL FOLDER DLP: {len(dlp_folders)}")

# =========================================================
# UPLOAD LOOP
# =========================================================
ia_success_count = 0
ia_fail_count = 0
ia_skip_count = 0
ia_no_pdf_count = 0

for i, (folder_name, folder_path) in enumerate(dlp_folders, start=1):

    print(f"\n================================================")
    print(f"[{i}/{len(dlp_folders)}] {folder_name}")

    # =====================================================
    # RESUME CHECK
    # =====================================================
    if folder_name in ia_uploaded_map and ia_uploaded_map[folder_name]:
        print(f"SKIP - SUDAH UPLOAD ({ia_uploaded_map[folder_name]})")
        ia_skip_count += 1
        continue

    # =====================================================
    # CARI PDF (skip kalau buku ini gagal diunduh di Cell 2,
    # jadi tidak punya PDF)
    # =====================================================
    pdf_path = None

    for fname in os.listdir(folder_path):
        if fname.lower().endswith(".pdf"):
            pdf_path = os.path.join(folder_path, fname)
            break

    if not pdf_path:
        print("SKIP - TIDAK ADA PDF")
        ia_no_pdf_count += 1
        continue

    print(f"PDF: {os.path.basename(pdf_path)}")
    print(f"SIZE: {os.path.getsize(pdf_path) / 1024 / 1024:.2f} MB")

    # =====================================================
    # BACA detail.json (skema notebook ini: title, keyword,
    # identifier, detail_url, txt_url, pdf_url, folder)
    # =====================================================
    detail_json_path = os.path.join(folder_path, "detail.json")
    detail = {}

    if os.path.exists(detail_json_path):
        try:
            with open(detail_json_path, "r", encoding="utf-8") as f:
                detail = json.load(f)
        except Exception as e:
            print(f"GAGAL BACA detail.json: {e}")

    title = detail.get("title") or folder_name
    keyword = detail.get("keyword", "")
    identifier_asal = detail.get("identifier", "")
    detail_url = detail.get("detail_url", "")

    # File pendukung lain yang ikut diupload kalau ada
    extra_files = [pdf_path]
    txt_path = os.path.join(folder_path, "file.txt")
    if os.path.exists(txt_path):
        extra_files.append(txt_path)
    jpg_zip_path = os.path.join(folder_path, "jpg.zip")
    if os.path.exists(jpg_zip_path):
        extra_files.append(jpg_zip_path)

    # =====================================================
    # BUAT IDENTIFIER ARCHIVE.ORG (unik per folder DLP-*)
    # =====================================================
    safe_folder = "".join(
        c if c.isalnum() or c in "-_" else "-"
        for c in folder_name
    )[:80].strip("-")

    ia_identifier = f"DLP-{safe_folder}"

    print(f"IA IDENTIFIER: {ia_identifier}")
    print(f"TITLE: {title}")

    # =====================================================
    # METADATA UNTUK ARCHIVE.ORG
    # =====================================================
    subject_parts = ["delpher", "dutch colonial", "aceh", "atjeh", "indonesia"]
    if keyword:
        subject_parts.insert(0, keyword)

    ia_metadata = {
        "title": title,
        "mediatype": "texts",
        "collection": "opensource",
        "subject": "; ".join(subject_parts),
        "language": "Dutch",
        "licenseurl": "https://creativecommons.org/licenses/by/4.0/",
        "description": (
            f"Source: Delpher (delpher.nl)\n"
            f"Original identifier: {identifier_asal}\n"
            f"Detail URL: {detail_url}\n"
            f"Keyword pencarian: {keyword}"
        ),
    }

    # =====================================================
    # UPLOAD
    # =====================================================
    try:
        print("UPLOADING...")

        r = ia.upload(
            ia_identifier,
            files=extra_files,
            metadata=ia_metadata,
            verbose=True,
            retries=3,
            retries_sleep=15,
            checksum=True,
        )

        all_ok = all(resp.status_code in [200, 201] for resp in r)

        if all_ok:
            url = f"https://archive.org/details/{ia_identifier}"
            print(f"SUCCESS: {url}")
            ia_uploaded_map[folder_name] = url
            ia_success_count += 1

            # Catat URL archive.org ke detail.json folder ini
            detail["archive_url"] = url
            with open(detail_json_path, "w", encoding="utf-8") as f:
                json.dump(detail, f, ensure_ascii=False, indent=2)

            # Catat juga ke process.txt biar kelihatan langsung
            with open(os.path.join(folder_path, "process.txt"), "a", encoding="utf-8") as f:
                f.write(f"\nARCHIVE.ORG: {url}")

            # Update entri buku ini di MASTER_DATA.json kalau ada
            with open(MASTER_JSON, "r", encoding="utf-8") as f:
                master_data = json.load(f)
            for entry in master_data:
                if entry.get("identifier") == identifier_asal:
                    entry["archive_url"] = url
            with open(MASTER_JSON, "w", encoding="utf-8") as f:
                json.dump(master_data, f, ensure_ascii=False, indent=2)
        else:
            print("FAILED - response not OK")
            ia_fail_count += 1

    except Exception as e:
        print(f"ERROR: {e}")
        ia_fail_count += 1
        continue

    # =====================================================
    # SAVE PROGRESS
    # =====================================================
    with open(IA_PROGRESS_FILE, "w") as f:
        json.dump({"uploaded": ia_uploaded_map}, f, indent=2, ensure_ascii=False)

    time.sleep(3)

# =========================================================
# SUMMARY
# =========================================================
print("\n================================================")
print("SELESAI UPLOAD KE INTERNET ARCHIVE")
print(f"SUCCESS  : {ia_success_count}")
print(f"SKIP     : {ia_skip_count}")
print(f"NO PDF   : {ia_no_pdf_count}")
print(f"FAILED   : {ia_fail_count}")
print(f"TOTAL    : {len(dlp_folders)}")
print("================================================")
print(f"Progress: {IA_PROGRESS_FILE}")
